In [69]:
import os
import json
import pandas
import numpy as np
import xml.etree.ElementTree as et

In [25]:
training_data_index = pandas.read_excel(os.path.join(os.getcwd(), "training_data_index.xlsx"))
training_data_index.head()

,expt,date,strain,well,qualitative_density_in_TC_dish,objective,duration_hr,interval_minutes,bench_score_immediate,Stitched?,Zarr?,Annotated?,Aggregates?,path_to_zarr,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18
0,mixin41,2024-01-09,NC28.1,B2,high,10x,42.0,10.0,fbs,yes,yes,yes,"first is high quality, rest are low quality",Y:\jennifer\cryolite\cryolite_mixin_test41_202...,NaN,NaN,NaN,NaN,NaN
1,mixin41,2024-01-09,NC4,B3,high,10x,42.0,10.0,fbs_culms_slugs_clumps,yes,yes,yes,"10, first few are high quality, rest are okay",Y:\jennifer\cryolite\cryolite_mixin_test41_202...,NaN,NaN,NaN,NaN,NaN
2,mixin42,2024-01-12,NC4,B2,medium,10x,66.0,10.0,mds_stalks,yes,yes,yes,"4 all kind of low quality, seems like it could...",Y:\jennifer\cryolite\cryolite_mixin_test42_202...,NaN,NaN,NaN,NaN,NaN
3,mixin42,2024-01-12,NC105.1,B3,medium,10x,66.0,10.0,mds_clumps,yes,yes,yes,"4, all kind of low quality",Y:\jennifer\cryolite\cryolite_mixin_test42_202...,NaN,NaN,NaN,NaN,NaN
4,mixin43,2024-01-16,NC28.1,B2,medium_low,10x,24.0,3.5,NaN,yes,yes,no,"no, just slugs",NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
experiment_paths = training_data_index['path_to_zarr'].tolist()
# remove nan items from list
experiment_paths = [path for path in experiment_paths if isinstance(path, str)]

In [29]:
measurement_points_stats_list = []
for dataset in experiment_paths:
    measurement_points_stats = pandas.read_csv(os.path.join(dataset, "measurement_points_Statistics", "measurement_points_Average.csv"))
    print(f"Loaded measurement points from {dataset}")
    measurement_points_stats_list.append(measurement_points_stats)
measurement_points_stats_list[0].head()

Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test41_2024-01-09\Field_B2
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test41_2024-01-09\Field_B3
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test42_2024-01-12\Field_B2
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test42_2024-01-12\Field_B3
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test44_2024-01-17
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test45_2024-01-30
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test49_2024-02-08
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test48_2024-02-08
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test52_2024-02-21
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test53_2025-02-22
Loaded measurement points from Y:\jennifer\cryolite\cryolite_mixin_test54_2024-02-22
Loaded measurement points fro

,Average,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,Variable,Min,Max,Mean,StdDev,Median,Percentile 25%,Percentile 75%,Sum,Count,Unit,Category,Channel,Collection,Image,Name,Time
1,Angle,53.957,53.957,53.957,0,53.957,53.957,53.957,53.957,1,°,MeasurementPoint,NaN,NaN,NaN,B,153
2,Angle,36.789,36.789,36.789,0,36.789,36.789,36.789,36.789,1,°,MeasurementPoint,NaN,NaN,NaN,C,253
3,Distance,782.128,782.128,782.128,0,782.128,782.128,782.128,782.128,1,µm,MeasurementPoint,NaN,NaN,NaN,A-B,NaN
4,Distance,685.199,685.199,685.199,0,685.199,685.199,685.199,685.199,1,µm,MeasurementPoint,NaN,NaN,NaN,B-C,NaN


In [66]:
coords_dataset = []

for measurement_points in measurement_points_stats_list:
    point_count = sum(1 for row in measurement_points.itertuples() if row.Average.startswith('Position X'))
    dataset_points = np.zeros((point_count, 4)).astype(int)
    # extract all rows that start with Position X
    x_rows = [row for row in measurement_points.itertuples() if row.Average.startswith('Position X')]
    y_rows = [row for row in measurement_points.itertuples() if row.Average.startswith('Position Y')]
    z_rows = [row for row in measurement_points.itertuples() if row.Average.startswith('Position Z')]
    print(f"Found {len(x_rows)} Position X rows in dataset")
    for i, row in enumerate(x_rows):
        dataset_points[i, 3] = int(round(float(row._2)))
        dataset_points[i, 0] = int(row._17)
    for i, row in enumerate(y_rows):
        dataset_points[i, 2] = int(round(float(row._2)))
    for i, row in enumerate(z_rows):
        dataset_points[i, 1] = int(round(float(row._2)))
    print(dataset_points)
    coords_dataset.append(dataset_points)

Found 4 Position X rows in dataset
[[ 48 208 709 891]
 [153   0 651 139]
 [253   0 142 598]
 [253   0 866 531]]
Found 9 Position X rows in dataset
[[ 70  10 479 851]
 [120   8 464 837]
 [131   0 303 531]
 [253   0 220 200]
 [190   0 103 539]
 [158   0 224 875]
 [253   6 715 760]
 [202   0 962 291]
 [ 84   2 897  23]]
Found 4 Position X rows in dataset
[[ 66   4 849 315]
 [267 178 752 241]
 [223   2 210 756]
 [344   0  52  66]]
Found 4 Position X rows in dataset
[[167   0 111 310]
 [397   8 121 930]
 [397   0  82 145]
 [397   0 101 460]]
Found 4 Position X rows in dataset
[[149 140 412 885]
 [205   8 158 318]
 [162 174 780 141]
 [105   0 373 713]]
Found 5 Position X rows in dataset
[[  45   88 1045  744]
 [  68  132  833  969]
 [  70  132  447  389]
 [  63  184 1635  781]
 [  67  132  947  163]]
Found 7 Position X rows in dataset
[[  69    6 1005 1363]
 [  91  150  631  828]
 [  74   10 1157 1714]
 [  75   14  947  844]
 [  96  138  452 1482]
 [ 132  136  354  154]
 [ 103   12 1070  697

In [87]:
def getSizeFromXML(xmlFile, res_lvl=0):
    XMLTree = et.parse(xmlFile)
    XMLRoot = XMLTree.getroot()
    imageMetaData = XMLRoot[res_lvl][0]
    SizeX = int(imageMetaData.get('SizeX'))
    SizeY = int(imageMetaData.get('SizeY'))
    SizeZ = int(imageMetaData.get('SizeZ'))
    SizeT = int(imageMetaData.get('SizeT'))
    return [SizeT, SizeZ, SizeY, SizeX]


In [79]:
zarr_paths = []
for path in experiment_paths:
    # find the directory that ends in .zarr
    zarr_dir = [d for d in os.listdir(path) if d.endswith('.zarr')]
    zarr_paths.append(os.path.join(path, zarr_dir[0]))
for path in zarr_paths:
    if not os.path.exists(path):
        print(f"Zarr directory not found: {path}")

metadata_paths = [os.path.join(path, "OME", "METADATA.ome.xml") for path in zarr_paths]
for path in metadata_paths:
    if not os.path.exists(path):
        print(f"Metadata file not found: {path}")

In [ ]:
# invert y coord and save as json
json_dataset = [dataset.tolist() for dataset in coords_dataset]

for metadata_path, dataset in zip(metadata_paths, json_dataset):
    dataset_shape = getSizeFromXML(metadata_path)
    print(f"Size Y = {dataset_shape[2]}")
    for coords in dataset:
        coords[2] = dataset_shape[2] - coords[2]  # Invert Y coordinate
        print(f"Coordinate point: {coords}")

Size Y = 977
Coordinate point: [48, 208, 709, 891]
Coordinate point: [153, 0, 651, 139]
Coordinate point: [253, 0, 142, 598]
Coordinate point: [253, 0, 866, 531]
Size Y = 978
Coordinate point: [70, 10, 479, 851]
Coordinate point: [120, 8, 464, 837]
Coordinate point: [131, 0, 303, 531]
Coordinate point: [253, 0, 220, 200]
Coordinate point: [190, 0, 103, 539]
Coordinate point: [158, 0, 224, 875]
Coordinate point: [253, 6, 715, 760]
Coordinate point: [202, 0, 962, 291]
Coordinate point: [84, 2, 897, 23]
Size Y = 978
Coordinate point: [66, 4, 849, 315]
Coordinate point: [267, 178, 752, 241]
Coordinate point: [223, 2, 210, 756]
Coordinate point: [344, 0, 52, 66]
Size Y = 978
Coordinate point: [167, 0, 111, 310]
Coordinate point: [397, 8, 121, 930]
Coordinate point: [397, 0, 82, 145]
Coordinate point: [397, 0, 101, 460]
Size Y = 977
Coordinate point: [149, 140, 412, 885]
Coordinate point: [205, 8, 158, 318]
Coordinate point: [162, 174, 780, 141]
Coordinate point: [105, 0, 373, 713]
Size Y = 

In [93]:
# create a dictionary with experiment_path as key and dataset as value
json_dictionary = {experiment_path: dataset for experiment_path, dataset in zip(experiment_paths, json_dataset)}
path_to_json = os.path.join(os.getcwd(), "training_data.json")
with open(path_to_json, 'w') as f:
    json.dump(json_dictionary, f)